# 任務 6：探索變數關係
目標：畫散布圖、計算相關係數並分辨關聯與因果。

**學生版**｜與教師版任務及小節編號一一對應

## 開始之前
**姓名：**何韋仁　**學號／組別：**C114118243　**完成日期：**2026/11/13


本教材可以在 Colab 或 Jupyter 執行，請依序執行儲存格。每份任務的準備儲存格會在記憶體重新建立相同的模擬資料，不會覆寫 CSV。若要分析自己先前產生的 CSV，先上傳檔案，執行準備儲存格後再執行 `df = pd.read_csv("student_learning_data.csv")`；不同資料的驗證數字可能不同。

成績是人為公式生成，資料品質問題在計分後加入。結果只描述模擬資料，不能推論真實學生或證明因果。每個任務請先預測，執行後修改與驗證，最後用自己的話解釋。

### 學生操作方式
1. 先執行下一個完整的資料準備儲存格。每份任務可獨立執行；重啟環境後請由上而下重新執行。
2. 在每個小節的程式中，將 `___` 替換成符合題意的程式碼，並把 `# 學生註解` 改成自己的繁體中文解說（至少說明指定程式的輸入／處理／輸出）。`___` 是刻意留下的空格，完成前執行會報錯。
3. 先記錄預測，執行後完成驗證與解釋；再完成延伸練習。遇到困難可以使用小節專屬 AI 提示詞，仍須檢查、修改、執行與理解。
4. 在文字儲存格中填寫答案，保留程式、執行結果、圖表和 AI 使用紀錄。依教師指示交付已完成的 `.ipynb`。首次執行任務 1 的存檔程式會覆寫同名 practice_student_data.csv，請先確認或改檔名。


In [ ]:
# 載入分析與繪圖套件；np、pd、plt 是慣用縮寫
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 固定種子及呼叫順序，使本課範例能重現
rng = np.random.default_rng(42)
n = 60  # 模擬學生人數
departments = ["資訊管理", "企業管理", "金融管理"]
# 字典每個鍵是欄位名稱，每個值提供 60 位學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)  # 把字典轉成資料表
noise = rng.normal(0, 5, n)  # 模擬其他未觀察的差異
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)  # 限制 0 到 100 分
df.loc[[5, 17], "sleep_hours"] = np.nan  # 成績算完後才加入缺失
df.loc[8, "screen_time_hours"] = 15.0  # 成績算完後才加入異常值
print("資料已準備：", df.shape)

## 6.1 有效配對
相關分析需要每位學生的兩個欄位都有值；缺失一方的資料無法形成有效配對。
**預測：** 睡眠與成績有幾組有效配對？

### 學生練習 6.1：完成程式與註解

- **程式填空：** 補上只排除 sleep_hours 或 quiz_score 缺失資料列的運算。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：**（請填寫）

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 6.1。我的資料表叫 df，請教我如何補上只排除 sleep_hours 或 quiz_score 缺失資料列的運算。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 學生註解 1：這段程式的輸入為原始 DataFrame `df`，處理為呼叫 `.dropna()` 方法並透過子集參數 `subset=["sleep_hours", "quiz_score"]` 指定只排除這兩個關鍵欄位中含有缺失值的整列資料，輸出為全新且無缺失值干擾的有效配對資料表 `pairs`。
pairs = df.dropna(subset=["sleep_hours", "quiz_score"])

print("有效配對數：", len(pairs))
print("原始列數：", len(df))

# 學生註解 2：利用 `assert` 斷言機制來快速進行資料完整性核對。由於已知僅有 2 筆睡眠資料缺失，因此預期有效配對數必須剛好等於 58 筆（原始 60 筆減去 2 筆），若條件不合則會拋出錯誤警告。
assert len(pairs) == 58, "⚠️ 警告：有效配對數目與預期不合！"
print("\n✅ 核對結果：有效配對數為 58 筆無誤，已成功排除 2 筆缺失值！")

### 驗證、解釋與延伸 6.1

**驗證與解釋：** 有效配對數及原始筆數各是多少？減少幾筆？

**我的結果與解釋（引用實際輸出）：**（請填寫）

**延伸練習：** 自行檢查配對表的兩個欄位各有多少缺失。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：**（請填寫）

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：自行檢查配對表的兩個欄位各有多少缺失。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 學生註解 1：這段程式的輸入為 `df` 與 `pairs`，處理方式為調用 `.isna().sum()` 方法，分別統計兩個資料表中特定欄位包含缺失值（NaN）的數量，最後將統計結果輸出於螢幕。
print("=== [原始資料表 df] 缺失值檢查 ===")
print("sleep_hours 缺失值數量:", df["sleep_hours"].isna().sum())
print("quiz_score 缺失值數量:", df["quiz_score"].isna().sum())

print("\n=== [配對資料表 pairs] 缺失值檢查 ===")
print("sleep_hours 缺失值數量:", pairs["sleep_hours"].isna().sum())
print("quiz_score 缺失值數量:", pairs["quiz_score"].isna().sum())

# 學生註解 2：此處利用 assert 斷言來確保篩選後的 `pairs` 表中，兩個關鍵欄位的缺失值數量完全為 0。這可以百分之百驗證我們清洗與排除缺失值步驟的正確性。
assert pairs["sleep_hours"].isna().sum() == 0, "⚠️ 警告：配對表中仍留有睡眠時數缺失值！"
assert pairs["quiz_score"].isna().sum() == 0, "⚠️ 警告：配對表中仍留有成績缺失值！"
print("\n✅ 核對結果：配對表中的兩個關鍵欄位缺失值皆為 0 筆，驗證成功！")

## 6.2 散布圖
每個點代表一位學生，橫軸是讀書時數，縱軸是成績。觀察方向、分散與特殊點。

### 學生練習 6.2：完成程式與註解

- **程式填空：** 補上以 study_hours 為橫軸、quiz_score 為縱軸的散布圖指令。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：**（請填寫）

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 6.2。我的資料表叫 df，請教我如何補上以 study_hours 為橫軸、quiz_score 為縱軸的散布圖指令。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 學生註解 1：這段填空程式的輸入是排除缺失值後的有效配對資料表 `study_pairs`，處理方式為調用 `plt.scatter()` 函數，並指定 `study_hours` 為橫軸變數、`quiz_score` 為縱軸變數，同時自訂點的顏色與邊框，輸出為呈現讀書時數與成績分布關係的二維散佈圖。
study_pairs = df.dropna(subset=["study_hours", "quiz_score"])
plt.scatter(study_pairs["study_hours"], study_pairs["quiz_score"], color="teal", edgecolor="black", alpha=0.8)

plt.xlabel("Study hours")
plt.ylabel("Quiz score")
plt.title("Study Hours and Quiz Score")

# 學生註解 2：`plt.show()` 這行程式的作用是向系統發出正式的畫布渲染與顯示指令，將我們前面設定好的散佈圖點群、X軸/Y軸標籤及大標題整合繪製並呈現於螢幕上。
plt.show()

# 核對與驗證
print("繪圖點數（有效筆數）：", len(study_pairs))
assert len(study_pairs) == 60, "⚠️ 警告：繪圖所使用的有效配對資料筆數不應少於 60 筆！"

### 驗證、解釋與延伸 6.2

**驗證與解釋：** 每個點代表什麼？大致方向與分散情形為何？

**我的結果與解釋（引用實際輸出）：**（請填寫）

**延伸練習：** 改用 sleep_hours 與 quiz_score 畫圖，記錄有效配對數。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：**（請填寫）

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：改用 sleep_hours 與 quiz_score 畫圖，記錄有效配對數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 1. 排除 sleep_hours 與 quiz_score 欄位中的缺失值並取得有效配對
# 學生註解 1：這段程式輸入為原始 df，處理為透過 dropna 排除 sleep_hours 及 quiz_score 含有 NaN 的列，輸出為 58 筆資料的獨立 DataFrame 副本 sleep_pairs。
sleep_pairs = df.dropna(subset=["sleep_hours", "quiz_score"]).copy()

# 2. 繪製睡眠時間與學期成績的散佈圖
plt.scatter(sleep_pairs["sleep_hours"], sleep_pairs["quiz_score"], color="coral", edgecolor="black", alpha=0.8)
plt.xlabel("Sleep hours")
plt.ylabel("Quiz score")
plt.title("Sleep Hours and Quiz Score")

# 學生註解 2：調用 plt.show() 正式對繪圖視窗發出繪製指令，將設定好的 X 軸、Y 軸標籤與標題呈現在畫布上。
plt.show()

# 3. 輸出並驗證有效配對數
print("睡眠與成績有效配對數：", len(sleep_pairs))
assert len(sleep_pairs) == 58, "⚠️ 警告：有效配對數不符合預期的 58 筆！"

## 6.3 Pearson 相關係數
相關係數介於 -1 和 1；正負表示線性方向，靠近 0 表示線性關係較弱。不要只靠一個數字判斷，應搭配散布圖與有效筆數。

### 學生練習 6.3：完成程式與註解

- **程式填空：** 補上讀書時數與成績的 Pearson 相關係數計算。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：**（請填寫）

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 6.3。我的資料表叫 df，請教我如何補上讀書時數與成績的 Pearson 相關係數計算。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 學生註解 1：這段填空程式的輸入是排除缺失值後的有效 DataFrame 數列 `study_pairs["study_hours"]`，處理方式為調用 `.corr()` 函數並傳入目標變數 `study_pairs["quiz_score"]` 計算 Pearson 線性相關係數，輸出為一個代表兩者線性相關強度的浮點數。
study_corr = study_pairs["study_hours"].corr(study_pairs["quiz_score"])
sleep_corr = pairs["sleep_hours"].corr(pairs["quiz_score"])

print("讀書配對數與相關：", len(study_pairs), study_corr)
print("睡眠配對數與相關：", len(pairs), sleep_corr)

# 學生註解 2：這行程式 `pairs["sleep_hours"].corr(pairs["quiz_score"])` 的作用是計算排除缺失值後，學生睡眠時數與期末成績之間的 Pearson 相關係數，讓我們能與讀書時間的相關係數進行對照，評估何者與成績的線性關聯性較強。

# 核對與驗證
assert -1 <= study_corr <= 1, "⚠️ 相關係數數值必須介於 -1 與 1 之間！"
assert study_corr > sleep_corr, "💡 核對：讀書時數的相關性通常應高於睡眠時數。"

### 驗證、解釋與延伸 6.3

**驗證與解釋：** 記錄兩組配對數與係數，正負號代表什麼？為何不能據此斷言因果？

**我的結果與解釋（引用實際輸出）：**（請填寫）

**延伸練習：** 計算 screen_time_hours 與 quiz_score 的相關係數和有效配對數。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：**（請填寫）

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：計算 screen_time_hours 與 quiz_score 的相關係數和有效配對數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 前置步驟提示：此練習將使用原始 df 變數，並排除兩欄中的缺失與異常值。

import pandas as pd
import numpy as np

# 1. 排除 screen_time_hours 與 quiz_score 中的缺失值以取得有效配對
# 學生註解 1：這段程式的輸入是原始 DataFrame `df`，處理方式為調用 `.dropna()` 排除 "screen_time_hours" 與 "quiz_score" 含有 NaN 的列，輸出為篩選後的有效配對資料表 `screen_pairs`。
screen_pairs = df.dropna(subset=["screen_time_hours", "quiz_score"]).copy()

# 2. 計算 Pearson 線性相關係數
# 學生註解 2：此行程式使用 `.corr()` 函數計算學生螢幕使用時間與期末成績之間的 Pearson 相關係數，幫助我們評估科技產品使用時間是否與成績存在線性關聯。
screen_corr = screen_pairs["screen_time_hours"].corr(screen_pairs["quiz_score"])

print("螢幕時間有效配對數：", len(screen_pairs))
print("螢幕時間與成績相關係數：", screen_corr)

# 核對與驗證
assert len(screen_pairs) == 60, "⚠️ 預期有效配對數應為 60 筆（因螢幕時間無 NaN 缺失值）。"
assert -1 <= screen_corr <= 1, "⚠️ 相關係數數值必須介於 -1 與 1 之間！"

## 任務交付與 AI 使用紀錄

完成本任務的三個小節，儲存含輸出的 Notebook，依教師指定的方式繳交。任務 8 另需完成 8.3 的完整分析報告及其佐證圖表。

- 使用的 AI 提示詞（未使用請填「未使用」）：
- AI 建議中我修改或拒絕的一處及原因：
- 遇到的錯誤或疑問、修正與驗證方式：
- 本任務一項有數據支持的發現及對應儲存格：
- 一項資料或推論限制：

**交付前檢核：** 已填完程式空格與註解；已填寫預測、驗證、解釋與延伸；已由上到下執行並確認圖表和數字與文字一致。
